# T2D Risk Screener — End-to-End Tutorial

**Authors:** Team 3c — Tyson Johnson, Alizeh Murtaza, Nithila Neethi Devan, Tariq Abdulhak (CDS-492, George Mason University)

This notebook reproduces the full pipeline for the Type 2 Diabetes (T2D) risk-screening model in one runnable document. It is the single starting point for anyone evaluating the project.

## Two modes

Set the `MODE` constant in the config cell below:

| Mode | What happens | Runtime |
|---|---|---|
| `"pretrained"` (default) | Loads the committed CatBoost bundle from `../models/` and evaluates it on a held-out test slice. | ~30 s on CPU |
| `"train"` | Retrains CatBoost from scratch with the exact hyperparameters from `train.py`, then evaluates. | ~5–10 min on CPU |

## What you will see

1. Printed test metrics — ROC-AUC, PR-AUC, Brier, sensitivity at the operating threshold.
2. ROC curve plot.
3. **Calibration (reliability) diagram** comparing raw CatBoost probabilities against Platt-calibrated ones. Saved to `../figures/calibration_pre_post.png`.
4. SHAP waterfall for one example patient.
5. A library-level prediction demo that goes from raw patient inputs through preprocessing, the calibrated model, and the SHAP explainer — the same code path the FastAPI server wraps.

The dataset (~91 MB) is downloaded to `../data/health_2024.csv` on first run and reused on subsequent runs.

## 1. Configuration

All toggles and paths live here. No magic numbers in code cells.

In [ ]:
from __future__ import annotations

# ---- User toggle ----------------------------------------------------------
MODE: str = "pretrained"   # options: "pretrained" | "train"

# ---- Reproducibility ------------------------------------------------------
SEED: int = 42

# ---- Paths (relative to the notebook, which lives in notebooks/) ---------
import pathlib

PROJECT_ROOT: pathlib.Path = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
MODELS_DIR: pathlib.Path = PROJECT_ROOT / "models"
DATA_DIR: pathlib.Path = PROJECT_ROOT / "data"
FIGURES_DIR: pathlib.Path = PROJECT_ROOT / "figures"
DATA_FILE: pathlib.Path = DATA_DIR / "health_2024.csv"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

# ---- Calibration plot settings -------------------------------------------
N_CAL_BINS: int = 10           # reliability diagram bins
CAL_PNG_DPI: int = 180         # presentation-quality

# ---- Ensure the project root is on the Python path for `src` imports -----
import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

assert MODE in {"pretrained", "train"}, f"MODE must be 'pretrained' or 'train' (got {MODE!r})"
print(f"MODE = {MODE}")
print(f"PROJECT_ROOT = {PROJECT_ROOT}")

## 2. Imports and seed

In [ ]:
import json
import random

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split

from src.config import (
    MODEL_FEATURES,
    MODEL_NOMINAL_FEATURES,
    RISK_TIER_LABELS,
    RISK_TIER_QUANTILES,
    TARGET_CAT,
)
from src.preprocessing import (
    create_blindness_features,
    create_features,
    create_missingness_features,
    create_panel_features,
    create_screening_features,
    download_dataset,
    load_raw_data,
    prepare_catboost_inputs,
    prepare_single_prediction,
    tidy_column_names,
    validate_dataframe,
)


def set_seed(seed: int = SEED) -> None:
    """Seed ``random`` and ``numpy`` for reproducibility.

    CatBoost accepts its own ``random_seed`` constructor argument, so we
    pass ``SEED`` there explicitly when we fit the model. sklearn routines
    receive ``random_state=SEED`` at call sites.
    """
    random.seed(seed)
    np.random.seed(seed)


set_seed()
print("Imports OK. Seed set.")

## 3. Load and preprocess the NHIS dataset

`download_dataset()` fetches the 2024 Korean NHIS one-million-row health examination sample from GitHub on first run and skips on subsequent runs. All cleaning, validation, and feature engineering reuse the functions in `src/preprocessing.py` — we do not reimplement them here.

In [ ]:
# Patch the data path used by src.preprocessing so the notebook writes into
# ./data/ regardless of where Python thinks the project root is.
import src.preprocessing as _pp
import src.config as _cfg
_cfg.DATA_DIR = DATA_DIR
_cfg.DATA_FILE = DATA_FILE
_pp.DATA_FILE = DATA_FILE

download_dataset(data_path=DATA_FILE)
df_raw = load_raw_data(data_path=DATA_FILE)
print(f"Raw rows: {len(df_raw):,}")

df = tidy_column_names(df_raw)
df = create_blindness_features(df)
df = validate_dataframe(df)
df = create_features(df)
df = create_screening_features(df)
df = create_panel_features(df)

missing_flag_cols = [
    "hemoglobin", "serum_creatinine", "serum_got_ast",
    "serum_gpt_alt", "gamma_gtp",
    "total_cholesterol", "triglycerides", "hdl_cholesterol", "ldl_cholesterol",
]
df = create_missingness_features(df, missing_flag_cols)

available = [f for f in MODEL_FEATURES if f in df.columns]
df_model = df[available + [TARGET_CAT]].copy()
df_model = df_model[df_model[TARGET_CAT].notna()].reset_index(drop=True)
print(f"Model-ready rows: {len(df_model):,}  |  features: {len(available)}")
print(f"Positive rate: {df_model[TARGET_CAT].astype(int).mean():.3%}")

## 4. Train / validation / test split

Stratified 60 / 20 / 20. Same seed and ratios as `train.py` so the `"pretrained"` path evaluates the committed model against the exact test slice it was evaluated on at training time.

In [ ]:
y = df_model[TARGET_CAT].astype(int)
X = df_model[available].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train, y_train, test_size=0.25, random_state=SEED, stratify=y_train
)

X_train_cb, X_valid_cb, X_test_cb, cat_indices = prepare_catboost_inputs(
    X_train[available], X_valid[available], X_test[available]
)
print(f"Split sizes  — train: {len(y_train):,}   valid: {len(y_valid):,}   test: {len(y_test):,}")

## 5. Load pretrained model OR train from scratch

Behaviour is controlled by `MODE`.

In [ ]:
def _prob_to_logit(prob: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    """Stable probability -> logit transform used for Platt scaling.

    Args:
        prob: Array of raw CatBoost probabilities in [0, 1].
        eps: Clip boundary to avoid log(0).

    Returns:
        Log-odds array with the same shape as ``prob``.
    """
    prob = np.clip(np.asarray(prob, dtype=float), eps, 1 - eps)
    return np.log(prob / (1 - prob))


def _apply_calibrator(cal: LogisticRegression, raw_prob: np.ndarray) -> np.ndarray:
    """Apply a Platt logistic calibrator to raw CatBoost probabilities."""
    logits = _prob_to_logit(raw_prob).reshape(-1, 1)
    return cal.predict_proba(logits)[:, 1]


if MODE == "pretrained":
    model = CatBoostClassifier()
    model.load_model(str(MODELS_DIR / "catboost_model.cbm"))
    calibrator = joblib.load(MODELS_DIR / "calibrator.pkl")
    meta = json.loads((MODELS_DIR / "metadata.json").read_text())
    threshold = float(meta["threshold"])
    print(f"Loaded pretrained model. Operating threshold = {threshold:.4f}")

else:  # MODE == "train"
    # Hyperparameters mirror train.py exactly
    ITERATIONS = 1500
    LEARNING_RATE = 0.03
    DEPTH = 6
    L2_LEAF_REG = 5.0
    TARGET_RECALL = 0.85

    pos = int(y_train.sum())
    neg = int(len(y_train) - pos)
    class_weights = [1.0, neg / max(pos, 1)]

    model = CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="AUC",
        iterations=ITERATIONS,
        learning_rate=LEARNING_RATE,
        depth=DEPTH,
        l2_leaf_reg=L2_LEAF_REG,
        random_seed=SEED,
        verbose=200,
        allow_writing_files=False,
        class_weights=class_weights,
    )
    model.fit(
        X_train_cb, y_train,
        cat_features=cat_indices,
        eval_set=(X_valid_cb, y_valid),
        use_best_model=True,
        early_stopping_rounds=100,
    )

    # Platt-scale on the validation set
    raw_valid = model.predict_proba(X_valid_cb)[:, 1]
    cal_logits = _prob_to_logit(raw_valid).reshape(-1, 1)
    calibrator = LogisticRegression(random_state=SEED).fit(cal_logits, y_valid)

    # Pick the lowest threshold that achieves TARGET_RECALL on validation
    from sklearn.metrics import precision_recall_curve
    cal_valid = _apply_calibrator(calibrator, raw_valid)
    precisions, recalls, thresholds = precision_recall_curve(y_valid, cal_valid)
    precisions, recalls = precisions[:-1], recalls[:-1]
    cands = np.where(recalls >= TARGET_RECALL)[0]
    idx = int(cands[np.argmax(precisions[cands])]) if len(cands) else int(np.argmax(recalls))
    threshold = float(thresholds[idx])
    print(f"Retrained from scratch. Selected threshold = {threshold:.4f}")

## 6. Evaluate on the held-out test slice

In [ ]:
raw_test = model.predict_proba(X_test_cb)[:, 1]
cal_test = _apply_calibrator(calibrator, raw_test)

roc_auc = roc_auc_score(y_test, cal_test)
pr_auc = average_precision_score(y_test, cal_test)
brier = brier_score_loss(y_test, cal_test)
y_pred = (cal_test >= threshold).astype(int)
sensitivity = (y_test[y_pred == 1].sum() / y_test.sum()) if y_test.sum() else 0.0

print(f"ROC-AUC               : {roc_auc:.4f}")
print(f"PR-AUC (AvgPrecision) : {pr_auc:.4f}")
print(f"Brier score           : {brier:.4f}")
print(f"Sensitivity @ t={threshold:.4f}: {sensitivity:.4f}")

### 6a. ROC curve

In [ ]:
fpr, tpr, _ = roc_curve(y_test, cal_test)

fig, ax = plt.subplots(figsize=(5.5, 5.0))
ax.plot(fpr, tpr, label=f"CatBoost (AUC = {roc_auc:.3f})", color="#2563eb", linewidth=2)
ax.plot([0, 1], [0, 1], "--", color="#9ca3af", linewidth=1, label="Chance")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("ROC Curve — Held-Out Test Set")
ax.legend(loc="lower right")
ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()

### 6b. Calibration (reliability) diagram — pre vs. post Platt scaling

Each point is one probability bin; its y-coordinate is the empirical positive rate in that bin. A well-calibrated model falls on the diagonal.

In [ ]:
frac_raw, mean_raw = calibration_curve(y_test, raw_test, n_bins=N_CAL_BINS, strategy="quantile")
frac_cal, mean_cal = calibration_curve(y_test, cal_test, n_bins=N_CAL_BINS, strategy="quantile")

fig, ax = plt.subplots(figsize=(6.0, 5.5))
ax.plot([0, 1], [0, 1], "--", color="#9ca3af", label="Perfect calibration", linewidth=1)
ax.plot(mean_raw, frac_raw, "o-", label="Raw", color="#ef4444", linewidth=2, markersize=6)
ax.plot(mean_cal, frac_cal, "s-", label="Platt-calibrated", color="#2563eb", linewidth=2, markersize=6)
ax.set_xlabel("Predicted probability")
ax.set_ylabel("Observed positive rate")
ax.set_title("Reliability Diagram — Pre vs. Post Platt Scaling")
ax.legend(loc="upper left")
ax.grid(alpha=0.2)
fig.tight_layout()

cal_png_path = FIGURES_DIR / "calibration_pre_post.png"
fig.savefig(cal_png_path, dpi=CAL_PNG_DPI, bbox_inches="tight")
print(f"Saved calibration plot -> {cal_png_path}")
plt.show()

## 7. SHAP waterfall for one example patient

We take the first row of the test set and use CatBoost's native `ShapValues` to decompose its prediction.

In [ ]:
sample = X_test_cb.iloc[[0]].copy()
pool = Pool(sample, cat_features=cat_indices)
shap_matrix = model.get_feature_importance(pool, type="ShapValues")
shap_vals = shap_matrix[0, :-1]
base_value = float(shap_matrix[0, -1])

top_k = 12
order = np.argsort(np.abs(shap_vals))[::-1][:top_k][::-1]
feature_names = list(sample.columns)
names = [feature_names[i] for i in order]
values = [float(shap_vals[i]) for i in order]
colors = ["#ef4444" if v > 0 else "#3b82f6" for v in values]

fig, ax = plt.subplots(figsize=(7.5, 5.5))
ax.barh(names, values, color=colors)
ax.axvline(0, color="#6b7280", linewidth=1)
ax.set_xlabel("SHAP value (log-odds contribution)")
ax.set_title(f"SHAP waterfall — sample patient (base log-odds = {base_value:+.3f})")
fig.tight_layout()
plt.show()

print(f"Raw CatBoost prob : {model.predict_proba(sample)[0, 1]:.4f}")
print(f"Calibrated prob   : {_apply_calibrator(calibrator, model.predict_proba(sample)[:, 1])[0]:.4f}")
print(f"True label        : {int(y_test.iloc[0])}")

## 8. End-to-end prediction demo — raw patient record in, calibrated probability + SHAP out

This is exactly the code path `POST /predict` runs inside the FastAPI server (`api/main.py`). We call the library functions directly to keep the notebook self-contained — no uvicorn thread needed.

In [ ]:
from src.model import ModelBundle, predict

# Wrap the already-loaded (or just-trained) artefacts in the same ModelBundle
# dataclass the API uses. This way the `predict()` call below hits the exact
# same inference path production uses.
bundle_meta = json.loads((MODELS_DIR / "metadata.json").read_text())

bundle = ModelBundle(
    model=model,
    calibrator=calibrator,
    threshold=threshold,
    tier_edges=bundle_meta["risk_tier_edges"],
    tier_labels=bundle_meta.get("risk_tier_labels", list(RISK_TIER_LABELS)),
    model_features=bundle_meta["model_features"],
    nominal_features=bundle_meta["nominal_features"],
    cat_feature_indices=bundle_meta["cat_feature_indices"],
)

# A realistic mid-50s patient with moderately elevated BP and a partial lipid panel
X_patient, _ = prepare_single_prediction(
    age=55,
    sex_code=1,
    height_cm=172.0,
    weight_kg=82.0,
    waist_cm=94.0,
    systolic_bp=134,
    diastolic_bp=86,
    urine_protein=1,
    smoking_status=2,
    alcohol_consumption=1,
    hemoglobin=15.1,
    serum_creatinine=0.9,
    ast=31.0,
    alt=34.0,
    ggt=48.0,
    total_cholesterol=212.0,
    triglycerides=188.0,
    hdl_cholesterol=44.0,
    ldl_cholesterol=None,  # deliberately missing to show graceful handling
    nominal_features=bundle.nominal_features,
    model_features=bundle.model_features,
)

result = predict(bundle, X_patient)
print(f"Calibrated probability : {result['probability']:.4f}")
print(f"Screen positive        : {result['screen_positive']}")
print(f"Risk tier              : {result['risk_tier']}")
print(f"Recommendation        : {result['recommendation']}")

## 9. What next?

- **Retrain for production** from a clean shell: `python train.py` at the project root. Outputs land in `models/`.
- **Serve the screener** to a browser: `uvicorn api.main:app --reload` and open `http://localhost:8000`.
- **Detailed metrics, comparisons, and caveats** — see `MODEL_CARD.md`.
- **Committed figures** (deck-ready) live in `figures/`.